# Lecture 4 · Recap the path from PDF to answer

> How many books can a student borrow at a time, for how long, and how can they renew them?

Use the [Library Borrowing Guide](../data/library-guide.pdf) for this review. You can complete it by reading the two pages. If you have outputs from the hands-on lectures, keep them open so you can point to the actual records behind your explanation.

[Lecture 1 · The question and scenario](lecture_1_introduction.ipynb) · [Lecture 2 · Build the index](lecture_2_pdf_to_index.ipynb) · [Lecture 3 · Retrieve and answer](lecture_3_retrieval_and_answering.ipynb)

## 1. Follow one fact through the system

Find **14 days** in the PDF. Before reading the table below, explain what happens to that fact while building the index and how a later question brings it into the model’s prompt.

| When | Follow the evidence |
| --- | --- |
| Index the guide | Docling reads the page-1 table as a table block. RAGAnything sends the table and nearby context to the text model for a description. The resulting table chunk retains a readable representation and source information. |
| Prepare search | The embedding model encodes the chunk text for similarity search. LightRAG also extracts entities and relationships, merges related records, and indexes descriptions of graph records. Each graph record keeps links to the chunks that support it. |
| Ask the question | In `mix` mode, the question searches for chunks directly. Low-level keywords also search for entities, and high-level keywords search for relationships. Graph neighbors and their supporting chunks can contribute evidence. |
| Read and answer | The system selects evidence within the context budget. The answer model receives instructions, readable evidence and the question. Check the loan period in the answer against the table, then check the renewal claims against page 2. |

**Recall from Lecture 2.** How does the diagram on page 2 enter the same search process? What happens to the ordinary paragraphs?

<details>
<summary>Check your explanation</summary>

The vision model describes the diagram. Its description can be embedded and retrieved as part of an image chunk. LightRAG divides ordinary text into overlapping token windows. A parsed block count therefore tells you little about the final number of chunks. You can inspect both the stored content and the vectors by chunk ID.

</details>

## 2. Audit a plausible answer

A student receives this reply: *“You can borrow three books for 14 days. You can always renew them for another week.”*

Underline each claim that the PDF supports, then rewrite the renewal sentence so a student could act on it. Use the page-1 table to check the numbers and page 2 to check eligibility, timing and steps.

<details>
<summary>Compare with a source-grounded answer</summary>

A student may borrow up to **3 books at a time for 14 days** (page 1). Before the due date, open **My Loans**, find the book and click **Renew**. If nobody else has reserved it, you may renew it **once for 7 extra days**. Check the new due date in My Loans. If someone has reserved the book, return it by the original due date (page 2).

“Always” contradicts the reservation condition. The table alone supports the first sentence. To support the second, the retrieval result must include evidence of the renewal rule. The diagram helps verify the action sequence.

</details>

**Trace the correction.** In Lecture 3’s retrieval output, find a chunk containing the reservation condition. Its `chunk_id` identifies the passage. A graph record’s `source_id` points to its supporting chunks. The answer’s `[1]` and the retrieved chunk’s `reference_id` identify the source file. Use the chunk’s page metadata where available and check the sentence on page 2 of the guide.

## 3. Predict what changes on the next query

Try each prediction before opening the explanations.

1. Using the **same index**, switch from `mix` to `naive`. Can both searches still find the correct numbers? What should you compare?
2. Run the ordinary text answer query again with the **same question and settings**. Which work can a keyword-cache hit save? Which work can an answer-cache hit save?
3. Ask “What steps does the renewal diagram show from left to right?” What would tell you the answering model actually received the image?

<details>
<summary>Compare your predictions</summary>

1. Both modes can find the borrowing table. The `naive` route searches chunk vectors. The `mix` route also uses graph retrieval. Inspect the passages for both the table values and the qualifying renewal sentence. A higher result count alone tells you little about coverage.
2. A keyword-cache hit supplies the saved search phrases and skips keyword generation. LightRAG can still embed inputs and retrieve evidence. An answer-cache hit then supplies the saved response and skips the final answer-model call. Index reuse lets both queries begin with the stored chunks, graph and vectors.
3. Check the number of images sent during that query. With `vlm_enhanced=False`, a correct answer may come from the indexed description alone. With `vlm_enhanced=True`, a positive image count confirms that a completed vision call received image data. If retrieval produces no usable image path, the system can fall back to a text answer.

</details>

To compare what the retrieval and generation stages produced, revisit the structured results, context, prompt preview and answer in Lecture 3. Each observation makes its own retrieval request. The ordinary answer call can serve an application on its own.

## 4. Transfer the lesson to a new borrower

A **teacher** asks for the maximum number of books they can borrow. The system retrieves the same student table. Draft a one-sentence reply. Then explain how a second guide containing the teacher policy could change your answer.

<details>
<summary>Check the scope of your evidence</summary>

“The available guide gives the student limit. I need a borrowing policy that covers teachers to establish your limit.”

Index the teacher policy alongside the student guide, then retrieve evidence for the teacher’s question. Facts from the two guides may connect through a shared entity such as Book. Each relationship still has its own supporting chunks. Verify that the passage you cite explicitly applies to teachers.

</details>

**Exit check.** Point to the source of each number in your student answer. For one of those numbers, name the stored passage, explain one search route that could find it, and show where a reader could verify it in the PDF.

The worked case uses Docling on one PDF containing body text, a table and a diagram. The second policy above is a thought experiment. Equations, scanned documents requiring OCR, alternative parsers, and questions with their own attachments are extensions of the example. The [full core workflow](rag_anything_core_workflow.ipynb) gives the detailed extraction walkthrough, retrieval modes and coverage notes.